In [ ]:
from google.colab import files

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print("Uploaded:", pdf_path)

# Extract all text lines from every page

In [ ]:
import pdfplumber

def get_lines(pdf_path):
    lines = []
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[0]
        words = page.extract_words()

        # group words into lines based on their vertical position
        words.sort(key=lambda w: (w["top"], w["x0"]))
        current_line = []
        current_top = None
        tolerance = 3  # pixels, words within this range count as same line

        for w in words:
            if current_top is None or abs(w["top"] - current_top) <= tolerance:
                current_line.append(w)
                current_top = w["top"] if current_top is None else current_top
            else:
                lines.append(current_line)
                current_line = [w]
                current_top = w["top"]
        if current_line:
            lines.append(current_line)

    result = []
    for line in lines:
        text = " ".join(w["text"] for w in line)
        x0 = min(w["x0"] for w in line)
        x1 = max(w["x1"] for w in line)
        top = min(w["top"] for w in line)
        bottom = max(w["bottom"] for w in line)
        result.append({"text": text, "x0": x0, "x1": x1, "top": top, "bottom": bottom})

    return result

lines = get_lines(pdf_path)
for l in lines:
    print(l)

# Build fillable fields across all pages

In [ ]:
import fitz

def make_fully_editable(input_path, output_path, lines):
    doc = fitz.open(input_path)
    page = doc[0]

    for i, line in enumerate(lines):
        rect = fitz.Rect(line["x0"], line["top"], line["x1"] + 20, line["bottom"] + 2)

        # cover the original printed text so it doesn't show twice
        page.draw_rect(rect, color=(1, 1, 1), fill=(1, 1, 1))

        widget = fitz.Widget()
        widget.field_name = f"field_{i}"
        widget.field_type = fitz.PDF_WIDGET_TYPE_TEXT
        widget.rect = rect
        widget.field_value = line["text"]
        widget.text_fontsize = 9
        page.add_widget(widget)

    doc.save(output_path)

output_path = "editable_" + pdf_path
make_fully_editable(pdf_path, output_path, lines)
print("Saved:", output_path)

In [ ]:
files.download(output_path)